In [ ]:
%%sql
SELECT DISTINCT
    id,
    name,
    is_cancelled,
    is_cancelled_by_patient,
    CASE
        WHEN is_cancelled = 1 AND is_cancelled_by_patient = 1 THEN 'Patient'
        WHEN is_cancelled = 1 AND is_cancelled_by_patient = 0 THEN 'VHG'
        ELSE 'Not Cancelled'
    END AS cancelled_by_category,
    CONCAT(
        TRIM(name),
        '_',
        CASE
            WHEN is_cancelled = 1 AND is_cancelled_by_patient = 1 THEN 'Patient'
            WHEN is_cancelled = 1 AND is_cancelled_by_patient = 0 THEN 'VHG'
            ELSE 'Not Cancelled'
        END
    ) AS expected_session_status_src_name
FROM silver_drj_appointment_attendances
ORDER BY id;

In [ ]:
%%sql
SELECT COUNT(*) AS total_rows
FROM (
    SELECT DISTINCT
        CONCAT(
            TRIM(name),
            '_',
            CASE
                WHEN is_cancelled = 1 AND is_cancelled_by_patient = 1 THEN 'Patient'
                WHEN is_cancelled = 1 AND is_cancelled_by_patient = 0 THEN 'VHG'
                ELSE 'Not Cancelled'
            END
        ) AS session_status_src_name
    FROM silver_drj_appointment_attendances
) x;

In [ ]:
CONCAT(
    TRIM(att.name),
    '_',
    CASE
        WHEN att.is_cancelled = 1 AND att.is_cancelled_by_patient = 1 THEN 'Patient'
        WHEN att.is_cancelled = 1 AND att.is_cancelled_by_patient = 0 THEN 'VHG'
        ELSE 'Not Cancelled'
    END
)

In [ ]:
CONCAT(
    'MPB001_',
    LOWER(TRIM(CONCAT(
        TRIM(att.name),
        '_',
        CASE
            WHEN att.is_cancelled = 1 AND att.is_cancelled_by_patient = 1 THEN 'Patient'
            WHEN att.is_cancelled = 1 AND att.is_cancelled_by_patient = 0 THEN 'VHG'
            ELSE 'Not Cancelled'
        END
    )))
)

In [ ]:
mpb_source AS (
    SELECT DISTINCT
        CONCAT(
            TRIM(att.name), '_',
            CASE
                WHEN att.is_cancelled = 1 AND att.is_cancelled_by_patient = 1 THEN 'Patient'
                WHEN att.is_cancelled = 1 AND att.is_cancelled_by_patient = 0 THEN 'VHG'
                ELSE 'Not Cancelled'
            END
        ) AS session_status_src_name,

        CONCAT(
            'MPB001_',
            LOWER(TRIM(CONCAT(
                TRIM(att.name), '_',
                CASE
                    WHEN att.is_cancelled = 1 AND att.is_cancelled_by_patient = 1 THEN 'Patient'
                    WHEN att.is_cancelled = 1 AND att.is_cancelled_by_patient = 0 THEN 'VHG'
                    ELSE 'Not Cancelled'
                END
            )))
        ) AS session_status_src_id,

        'MPB001' AS session_status_src_sys_inst_id
    FROM silver_drj_appointment_attendances att
    WHERE att.name IS NOT NULL
      AND TRIM(att.name) <> ''
)

newww

In [ ]:
    SELECT DISTINCT
        CONCAT(
            TRIM(att.name), '_',
            CASE
                WHEN att.is_cancelled = true AND att.is_cancelled_by_patient = true THEN 'Patient'
                WHEN att.is_cancelled = true AND att.is_cancelled_by_patient = false THEN 'VHG'
                ELSE 'Not Cancelled'
            END
        ) AS session_status_src_name,

        CONCAT(
            'MPB001_',
            CAST(att.id AS STRING), '_',
            CAST(CASE WHEN att.is_cancelled = true THEN 1 ELSE 0 END AS STRING), '_',
            CAST(CASE WHEN att.is_cancelled_by_patient = true THEN 1 ELSE 0 END AS STRING)
        ) AS session_status_src_id,

        'MPB001' AS session_status_src_sys_inst_id
    FROM silver_drj_appointments a
    LEFT JOIN silver_drj_appointment_attendances att
        ON a.attendance_id = att.id
    WHERE att.name IS NOT NULL
      AND TRIM(att.name) <> ''

In [ ]:
Yes, got it. The ADD output is correct now. For the master sessions join, I’ll need to join silver_drj_appointment_attendances using appt.attendance_id = att.id so we can recreate the same MPB001_<att.id>_<is_cancelled>_<is_cancelled_by_patient> key for the RDM join.